In [ ]:
# Core imports for the whole lab
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

sns.set_theme(style='whitegrid')   # nicer default plot styling
np.random.seed(42)                 # reproducible results
print('Setup complete. NumPy', np.__version__, '| Pandas', pd.__version__)


In [ ]:
# -----------------------------------------------------------
# 🔹 1A. ARRAY CREATION & SHAPE
# -----------------------------------------------------------

# arange() builds a range; reshape() changes the shape without copying data
a = np.arange(6).reshape(2, 3)
print('Array a:\n', a)

# Inspect the array's structure
print('shape :', a.shape)     # (2, 3)  -> 2 rows, 3 cols
print('ndim  :', a.ndim)      # 2       -> number of dimensions
print('dtype :', a.dtype)     # int64   -> data type of elements


In [ ]:
# -----------------------------------------------------------
# 🔹 1B. SLICING & INDEXING
# -----------------------------------------------------------

print('Column index 1 :', a[:, 1])        # all rows, 2nd column
print('Row index 0    :', a[0, :])        # 1st row, all columns

# Boolean mask: keep only the elements that satisfy a condition
print('Elements > 2   :', a[a > 2])

In [ ]:
# -----------------------------------------------------------
# 🔹 1C. BROADCASTING
# -----------------------------------------------------------

# Broadcasting lets NumPy combine arrays of different shapes without loops
row = np.array([10, 20, 30])   # shape (3,)
print('a + row (row added to every row of a):\n', a + row)


In [ ]:
# -----------------------------------------------------------
# 🔹 1D. MATRIX MULTIPLICATION + AXIS-WISE OPERATIONS  (core demo)
# -----------------------------------------------------------

b = np.ones((3, 2), dtype=int)   # shape (3, 2)
print('Matrix product a @ b:\n', a @ b)   # (2,3) @ (3,2) -> (2,2)

# Axis-wise reductions: axis=0 collapses rows, axis=1 collapses columns
print('Column-wise sum  (axis=0):', a.sum(axis=0))
print('Row-wise sum     (axis=1):', a.sum(axis=1))
print('Row-wise mean    (axis=1):', a.mean(axis=1))
print('Index of max per column  :', a.argmax(axis=0))

In [ ]:
import numpy as np

# Given data
M = np.arange(1, 13).reshape(3, 4)   # shape (3, 4)
N = np.arange(1, 9).reshape(4, 2)    # shape (4, 2)
print('M:\n', M)
print('N:\n', N)
print()

# 1. Matrix product M @ N
product = M @ N
print("Matrix Product (M @ N):\n", product)
print()

# 2. Column-wise sum (axis=0) and row-wise mean (axis=1) of M
col_sum = M.sum(axis=0)
row_mean = M.mean(axis=1)
print("Column-wise sum of M:", col_sum)
print("Row-wise mean of M:", row_mean)
print()

# 3. Index of the max value in each row of M
max_indices = M.argmax(axis=1)
print("Index of max value in each row of M:", max_indices)


In [ ]:
# -----------------------------------------------------------
# 🔹 2A. LOAD & INSPECT
# -----------------------------------------------------------

# Load the classic Titanic dataset (built into seaborn).
# In your own work this would be: pd.read_csv('titanic.csv')
df = sns.load_dataset('titanic')

print('Shape:', df.shape)
df.head()


In [ ]:
# .info() shows columns, dtypes and non-null counts -> spot missing data fast
df.info()


In [ ]:
# .describe() gives summary statistics for the numeric columns
df.describe()


In [ ]:
# -----------------------------------------------------------
# 🔹 2B. HANDLE MISSING VALUES
# -----------------------------------------------------------

# Count missing values per column
print('Missing values per column:')
print(df.isna().sum().sort_values(ascending=False).head())

# Fill missing 'age' with the median (a common, robust strategy)
df['age'] = df['age'].fillna(df['age'].median())
print('\nMissing ages after fillna:', df['age'].isna().sum())

In [ ]:
# -----------------------------------------------------------
# 🔹 2C. GROUP BY A CATEGORY  (split -> apply -> combine)
# -----------------------------------------------------------

# Average survival rate by passenger class
print('Survival rate by class:')
print(df.groupby('class', observed=True)['survived'].mean())

# Two-level grouping: survival by class AND sex
print('\nSurvival rate by class and sex:')
print(df.groupby(['class', 'sex'], observed=True)['survived'].mean())


In [18]:
import seaborn as sns

lab = sns.load_dataset('titanic')   # fresh copy to work on

# 1. Count missing values in 'embarked'
missing_embarked = lab['embarked'].isna().sum()
print(f"Missing values in 'embarked': {missing_embarked}\n")

# 2. Fill missing 'embarked' with the mode
embarked_mode = lab['embarked'].mode()[0]
lab['embarked'] = lab['embarked'].fillna(embarked_mode)
print(f"Filled missing values with mode: '{embarked_mode}'\n")

# 3. Mean fare grouped by 'sex'
mean_fare_by_sex = lab.groupby('sex', observed=False)['fare'].mean()
print("Mean fare grouped by sex:")
print(mean_fare_by_sex)


Missing values in 'embarked': 2

Filled missing values with mode: 'S'

Mean fare grouped by sex:
sex
female    44.479818
male      25.523893
Name: fare, dtype: float64


In [ ]:
# -----------------------------------------------------------
# 🔹 3A. SUBPLOT COMPARISON  (Matplotlib)
# -----------------------------------------------------------

iris = sns.load_dataset('iris')   # 150 flowers, 4 measurements, 3 species

# plt.subplots() makes a grid of axes for side-by-side charts
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(data=iris, x='petal_length', hue='species', ax=axes[0])
axes[0].set_title('Petal length distribution')

sns.scatterplot(data=iris, x='petal_length', y='petal_width',
                hue='species', ax=axes[1])
axes[1].set_title('Petal length vs width')

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------
# 🔹 3B. CORRELATION HEATMAP  (Seaborn)
# -----------------------------------------------------------

# .corr() on the numeric columns, visualised as a heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(iris.drop(columns='species').corr(),
            annot=True, cmap='Blues')
plt.title('Feature correlation')
plt.show()

In [ ]:
# -----------------------------------------------------------
# 🔹 3C. PAIRPLOT  (every feature vs every feature)
# -----------------------------------------------------------

# A pairplot draws scatter plots for all feature pairs at once,
# coloured by class -> a fast way to see how separable the classes are.
sns.pairplot(iris, hue='species', height=1.8)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

pen = sns.load_dataset('penguins')

# 1. Drop rows with missing values
pen = pen.dropna()

# 2. Pairplot coloured by 'species'
sns.pairplot(pen, hue='species')
plt.show()

# 3. 1x2 subplot: boxplot (left) + scatter (right)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left subplot: Boxplot
sns.boxplot(ax=axes[0], data=pen, x='species', y='body_mass_g')
axes[0].set_title('Body Mass by Species')

# Right subplot: Scatter plot
sns.scatterplot(ax=axes[1], data=pen, x='bill_length_mm', y='flipper_length_mm', hue='species')
axes[1].set_title('Bill Length vs Flipper Length')

plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------------------------------------
# 🔹 4A. LOAD DATA & SPLIT
# -----------------------------------------------------------

# Load iris as feature matrix X and target vector y
X, y = load_iris(return_X_y=True)
print('Features X:', X.shape, '| Target y:', y.shape)

# Hold out 20% of the data for testing. stratify keeps class balance.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape[0], 'samples | Test:', X_test.shape[0], 'samples')

In [ ]:
# -----------------------------------------------------------
# 🔹 4B. FIT -> PREDICT -> EVALUATE  (the estimator API)
# -----------------------------------------------------------

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)               # learn from training data
pred = clf.predict(X_test)              # score new samples

print('Accuracy:', round(accuracy_score(y_test, pred), 3))
print('Confusion matrix:\n', confusion_matrix(y_test, pred))

In [ ]:
# -----------------------------------------------------------
# 🔹 4C. PIPELINE  (preprocessing + model as one object)
# -----------------------------------------------------------

# A Pipeline chains steps so the SAME transform runs on train and test
# data -> no leakage, and a single .fit() / .predict() call.
pipe = Pipeline([
    ('scaler', StandardScaler()),               # step 1: scale features
    ('model', LogisticRegression(max_iter=200)) # step 2: classify
])

pipe.fit(X_train, y_train)
print('Pipeline accuracy:', round(pipe.score(X_test, y_test), 3))


In [ ]:

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

Xb, yb = load_breast_cancer(return_X_y=True)
print('Breast cancer data:', Xb.shape)

# 1. train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    Xb, yb, test_size=0.25, random_state=0, stratify=yb
)

# 2. Build a Pipeline (StandardScaler + a classifier)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=0))
])

# 3. Fit and print the test accuracy
pipe.fit(X_train, y_train)
test_accuracy = pipe.score(X_test, y_test)

print(f"Test Accuracy: {test_accuracy:.4f}")


In [ ]:
# -----------------------------------------------------------
# 🔹 5. END-TO-END MINI WORKFLOW
# -----------------------------------------------------------

# RAW  -> load the table (Pandas)
data = sns.load_dataset('titanic')

# CLEAN -> keep a few features, fill/drop missing, encode 'sex' (Pandas + NumPy)
data = data[['survived', 'pclass', 'sex', 'age', 'fare']].copy()
data['age'] = data['age'].fillna(data['age'].median())
data['sex'] = np.where(data['sex'] == 'male', 0, 1)   # encode to numbers
data = data.dropna()

# SPLIT -> features X and target y as NumPy arrays (NumPy)
X = data.drop(columns='survived').values
y = data['survived'].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# MODEL -> fit + predict + score (scikit-learn)
model = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)
acc = accuracy_score(y_te, model.predict(X_te))

# OUTPUT -> the result, plus a quick feature-importance chart (Matplotlib)
print('End-to-end Titanic survival accuracy:', round(acc, 3))

importances = pd.Series(model.feature_importances_,
                        index=data.drop(columns='survived').columns)
importances.sort_values().plot.barh(title='Feature importance')
plt.tight_layout(); plt.show()



In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import seaborn as sns

# 1. Load
df = sns.load_dataset('iris')

# 2. Clean/Encode
X = df.drop(columns='species')
encoder = LabelEncoder()
y = encoder.fit_transform(df['species'])

# 3. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Model
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train, y_train)

# 5. Output (print accuracy)
accuracy = model.score(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")


Test Accuracy: 0.9667
